# Q-learning and SARSA: lab notebook

**Machine Learning (5CS4-05), Semi-supervised and reinforcement learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np

### 1. Cliff walking: 4 x 8 grid, START bottom-left, GOAL bottom-right, CLIFF between them

In [ ]:
# 1. Cliff walking: 4 x 8 grid, START bottom-left, GOAL bottom-right, CLIFF between them
R, C = 4, 8
start, goal = 3 * C, 3 * C + 7
cliff = set(range(3 * C + 1, 3 * C + 7))
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
ARROW = "^>v<"

def step(s, a):
    r, c = s // C + ACTIONS[a][0], s % C + ACTIONS[a][1]
    s2 = s if not (0 <= r < R and 0 <= c < C) else r * C + c
    if s2 in cliff:
        return start, -100.0, False                    # fall: back to START
    return s2, -1.0, s2 == goal

def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(4))
    best = np.flatnonzero(Q[s] >= Q[s].max() - 1e-12)
    return int(rng.choice(best))

### 2. One training run, for either algorithm

In [ ]:
# 2. One training run, for either algorithm
def train(algo, episodes=500, alpha=0.2, eps=0.1, gamma=1.0, seed=4):
    rng = np.random.default_rng(seed)
    Q = np.zeros((R * C, 4)); rewards = []
    for _ in range(episodes):
        s = start; a = eps_greedy(Q, s, eps, rng); total = 0
        for _ in range(500):
            s2, r, done = step(s, a); total += r
            a2 = eps_greedy(Q, s2, eps, rng)
            nxt = 0.0 if done else (Q[s2, a2] if algo == "sarsa" else Q[s2].max())
            Q[s, a] += alpha * (r + gamma * nxt - Q[s, a])       # TD update
            s, a = s2, a2
            if done:
                break
        rewards.append(total)
    return Q, np.array(rewards)

def greedy_path(Q):
    s, path = start, [start]
    while s != goal and len(path) < 40:
        s, _, _ = step(s, int(np.argmax(Q[s]))); path.append(s)
    return path

### 3. Compare Q-learning and SARSA

In [ ]:
# 3. Compare Q-learning and SARSA
for algo in ("q", "sarsa"):
    Q, rew = train(algo)
    path = greedy_path(Q)
    rows_used = sorted({p // C for p in path})
    print(f"{algo:5s}: mean online reward (last 100) = {rew[-100:].mean():7.1f}, greedy path = {len(path) - 1} steps, rows visited = {rows_used}")

### 4. The greedy policy of each algorithm

In [ ]:
# 4. The greedy policy of each algorithm
for algo in ("q", "sarsa"):
    Q, _ = train(algo)
    print(algo, "greedy policy:")
    for i in range(R):
        print(" ".join("C" if i * C + j in cliff else "G" if i * C + j == goal else ARROW[int(np.argmax(Q[i * C + j]))] for j in range(C)))

### 5. With no exploration both agree

In [ ]:
# 5. With no exploration both agree
for algo in ("q", "sarsa"):
    Q, rew = train(algo, eps=0.0)
    print(f"{algo:5s} eps = 0: greedy path length", len(greedy_path(Q)) - 1)

## Expected output

Running all cells above should print:

```
q    : mean online reward (last 100) =   -29.8, greedy path = 9 steps, rows visited = [2, 3]
sarsa: mean online reward (last 100) =   -16.1, greedy path = 11 steps, rows visited = [1, 2, 3]
q greedy policy:
< > ^ < > > v v
> ^ > > v v > v
> > > > > > > v
^ C C C C C C G
sarsa greedy policy:
< v > > > > > v
> > > > > > > v
^ ^ ^ > ^ < > v
^ C C C C C C G
q     eps = 0: greedy path length 9
sarsa eps = 0: greedy path length 9
```

## Practice exercises

Try these in the cells below.

1. Set epsilon to 0 in the script. Does SARSA still take the safe path? Why or why not?
2. Decay epsilon from 0.5 to 0.01 over training and compare the final greedy paths.
3. Average 20 independent runs and plot reward per episode for both algorithms.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
